<a href="https://colab.research.google.com/github/LeMaterial/lematerial-llm-synthesis/blob/main/examples/notebooks/tutorials/04_extracting_synthesis_and_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 4 — From a PDF to synthesis + performance data

This is the full pipeline on one real paper, end to end: PDF in, structured
recipes and digitised performance curves out.

Everything runs against a **fixed example paper** so that the outputs in this
notebook are the ones you should get, and so the two halves of the pipeline
share their intermediate results instead of paying for them twice.

> **The example paper**
> *Enhanced Hydrogen Evolution Activity of MoS₂-rGO Composite Synthesized via
> Hydrothermal Technique* — Sebastian & Pragna R,
> [arXiv:2404.08872](https://arxiv.org/abs/2404.08872) (CC BY-NC-ND 4.0).
>
> It was picked deliberately: it synthesises exactly **two** materials (MoS₂ and
> a MoS₂–rGO composite) by the same hydrothermal route, it reports **LSV curves**
> we can digitise, and it is one of the 36 papers in this repository's
> `annotations/` corpus — so a domain expert has already written down the
> correct answer, and we can check our extraction against it for free.

## What you'll learn

1. Turning a PDF into markdown, with a local or an API-based extractor
2. Finding the synthesised materials, and extracting a structured recipe for each
3. Scoring those recipes with an LLM judge, then checking them against a human
   ground truth
4. Segmenting the figures and reading data points off the plots with a VLM
5. Filtering plots by domain and linking each curve back to the material that
   produced it
6. Saving results in the same layout the CLI produces

## Cost, and how this notebook avoids it

Every LLM result is **cached to disk** the first time it is computed. Re-running
the notebook — or restarting the kernel and running it again — reuses the cache
and costs nothing. Set `FORCE_REFRESH = True` in the configuration cell to
recompute deliberately.

A full first run is roughly **$0.10–0.40** depending on the models you choose:
one OCR pass, one material-extraction call, two synthesis calls, two judge
calls, one VLM call per figure, then linking and a linking judge.
Set `SKIP_PERFORMANCE = True` to stop after the synthesis half, which is by far
the cheaper one.

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- API keys in `.env` — see Step 0
- **Runtime:** 10–20 min for a cold run, seconds once cached

## Setup — local or Colab

This notebook runs unchanged in two places:

- **Locally**, from a clone of the repository (`uv sync && uv pip install -e .`),
  with your API keys in the `.env` file at the repository root.
- **On [Google Colab](https://colab.research.google.com)** — click the badge at
  the top. The cell below clones the repository and installs it, which takes a
  few minutes the first time, then reads your keys from Colab's **secret
  manager**: open the 🔑 icon in the left sidebar, add one secret per key
  (`GEMINI_API_KEY`, `HF_TOKEN`, …) and switch *Notebook access* on for each.

Either way the keys land in `os.environ` and nothing else in the notebook
changes — no key is ever passed as a function argument, so none of them can end
up in the notebook's output or in git.

> If an import fails immediately after the setup cell on Colab, use
> **Runtime → Restart session** and run it again: the clone is cached, so the
> second run is quick.


In [ ]:
# --- Setup: this cell is the only difference between local and Colab -----
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Every key the project knows about. A tutorial only needs a subset; whichever
# ones are missing are reported by the key check further down.
API_KEY_NAMES = (
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "MISTRAL_API_KEY",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
    "HF_TOKEN",
)

if IN_COLAB:
    REPO_URL = "https://github.com/LeMaterial/lematerial-llm-synthesis.git"
    REPO_ROOT = Path("/content/lematerial-llm-synthesis")

    if not REPO_ROOT.exists():
        print("Cloning the repository ...")
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_ROOT)],
            check=True,
        )
    print("Installing llm-synthesis (a few minutes on the first run) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_ROOT)],
        check=True,
    )

    # Colab keeps secrets outside the notebook, so they cannot leak into its
    # output: add them under the key icon in the left sidebar.
    from google.colab import userdata

    for name in API_KEY_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set, or notebook access not granted - reported below
    KEY_SOURCE = "Colab secrets"
else:
    from dotenv import find_dotenv, load_dotenv

    def find_repo_root(start: Path | None = None) -> Path:
        """Walk up from `start` (default: cwd) until a directory has pyproject.toml."""
        here = (start or Path.cwd()).resolve()
        for candidate in (here, *here.parents):
            if (candidate / "pyproject.toml").exists():
                return candidate
        raise RuntimeError(f"No pyproject.toml found above {here}")

    REPO_ROOT = find_repo_root()
    # find_dotenv walks up from the working directory, so this works whether you
    # started Jupyter at the repo root or inside this folder.
    env_path = find_dotenv(usecwd=True)
    load_dotenv(env_path, override=True)
    KEY_SOURCE = env_path or "no .env found"

print(f"environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f"repo root:   {REPO_ROOT}")
print(f"API keys:    {KEY_SOURCE}")


## Step 0a — Choose your models

Two ways to reach the models, chosen with a single flag:

- **Direct** (default) — one key per provider (`GEMINI_API_KEY`,
  `ANTHROPIC_API_KEY`). Model names are aliases resolved through `LLM_REGISTRY`
  in `src/llm_synthesis/utils/llms.py`.
- **OpenRouter** — one `OPENROUTER_API_KEY` for everything, and any model id
  from [openrouter.ai/models](https://openrouter.ai/models). Useful if you would
  rather not hold accounts with three providers, or want an open-weight model.

Set `USE_OPENROUTER = True` below and everything downstream follows; no other
cell changes.

In [ ]:
from pathlib import Path

# ==============================================================================
# USER CONFIGURATION
# ==============================================================================

# --- Provider ---------------------------------------------------------------
USE_OPENROUTER = True  # True routes every LLM call through OpenRouter

OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Model per pipeline role. Left column: aliases from LLM_REGISTRY (direct APIs).
# Right column: any model id from https://openrouter.ai/models.
DIRECT_MODELS = {
    "material": "gemini-3.0-flash",
    "synthesis": "gemini-3.0-flash",
    "judge": "gemini-3.0-flash",
    "linker": "gemini-3.0-flash",
    "vlm": "claude-sonnet-4-6",
}

OPENROUTER_MODELS = {
    "material": "google/gemini-3-flash-preview",
    "synthesis": "google/gemini-3-flash-preview",
    "judge": "google/gemini-3-flash-preview",
    "linker": "google/gemini-3-flash-preview",
    "vlm": "anthropic/claude-sonnet-4-6",
}

# Note on the "material" role: `gemini-3.0-pro` is noticeably better at
# spotting every distinct composition (dopant levels, loadings, composite
# variants) and at not inventing ones that were only cited. We keep flash here
# because it is cheap and sufficient for this two-material paper - switch the
# "material" entry to "gemini-3.0-pro" (or "google/gemini-3-pro-preview" on
# OpenRouter) for papers with many closely related samples.

# --- Pipeline ---------------------------------------------------------------
SKIP_PERFORMANCE = False  # True = synthesis only, no figures, no VLM calls
FORCE_REFRESH = False  # True = ignore the cache and re-call the models

# --- The example paper ------------------------------------------------------
PAPER_ID = "2404.08872"
PAPER_PDF_URL = "https://arxiv.org/pdf/2404.08872v1"

PDF_EXTRACTOR = "mistral"  # "mistral" (API, better OCR) or "docling" (local)

## Step 0b — API keys and your `.env` file

Nothing in this project takes an API key as a function argument. Keys live in a
single `.env` file at the repository root, get loaded into the process
environment once per session, and LiteLLM reads them from there. Your keys never
appear in notebook code, notebook output, or git history.

From the repository root:

```bash
cp .env.example .env
```

Then fill in the keys you need — one per line, no quotes, no spaces around `=`.

### Direct providers (`USE_OPENROUTER = False`)

| Key | What it unlocks | Needed? |
|-----|-----------------|---------|
| `GEMINI_API_KEY` | Material + synthesis extraction, linking, judges | **yes** |
| `ANTHROPIC_API_KEY` | Claude vision, reads data points off the plots | yes, unless `SKIP_PERFORMANCE` |
| `MISTRAL_API_KEY` | Mistral OCR for the PDF | only with `PDF_EXTRACTOR = "mistral"` |

### OpenRouter (`USE_OPENROUTER = True`)

| Key | What it unlocks | Needed? |
|-----|-----------------|---------|
| `OPENROUTER_API_KEY` | Every LLM **and** VLM call in this notebook | **yes** |
| `MISTRAL_API_KEY` | Mistral OCR — OCR does not go through OpenRouter | only with `PDF_EXTRACTOR = "mistral"` |

Get keys at [aistudio.google.com](https://aistudio.google.com/app/apikey)
(Gemini, free tier is enough), [console.anthropic.com](https://console.anthropic.com/),
[console.mistral.ai](https://console.mistral.ai/) and
[openrouter.ai/keys](https://openrouter.ai/keys).

The next cell prints only each key's *length*, never its value, so the output is
safe to share.

> **On Colab you do not need a `.env` file.** The setup cell above already
> read your keys from Colab's secret manager — add them there instead (🔑 in
> the left sidebar), with *Notebook access* switched on.


In [ ]:
import os
import logging

# dspy warns whenever .forward(...) is called directly instead of module(...);
# every extractor/judge in this pipeline is invoked via .forward() by design
# (see transformers/base.py's ExtractorInterface), so this is expected noise.
logging.getLogger("dspy.primitives.module").setLevel(logging.ERROR)

if USE_OPENROUTER:
    REQUIRED_KEYS = {"OPENROUTER_API_KEY": "every LLM and VLM call"}
else:
    REQUIRED_KEYS = {"GEMINI_API_KEY": "extraction, linking and judges"}
    if not SKIP_PERFORMANCE:
        REQUIRED_KEYS["ANTHROPIC_API_KEY"] = "plot data extraction (VLM)"

OPTIONAL_KEYS = {"MISTRAL_API_KEY": 'Mistral OCR (PDF_EXTRACTOR = "mistral")'}
if PDF_EXTRACTOR == "mistral":
    REQUIRED_KEYS.update(OPTIONAL_KEYS)
    OPTIONAL_KEYS = {}


def report_keys(required, optional):
    """Print which API keys the environment provided, never their values."""
    print(f"API keys from: {KEY_SOURCE}")
    print(f"provider: {'OpenRouter' if USE_OPENROUTER else 'direct APIs'}\n")
    missing = []
    for name, purpose in {**required, **optional}.items():
        value = os.getenv(name)
        is_required = name in required
        if value:
            status = f"set ({len(value)} chars)"
        elif is_required:
            status = "MISSING"
            missing.append(name)
        else:
            status = "not set"
        tag = "required" if is_required else "optional"
        print(f"  {name:<24} {status:<16} [{tag}] {purpose}")
    if missing:
        fix = (
            "Add them in Colab's secret manager (the key icon in the left "
            "sidebar) and switch on notebook access, then re-run this cell."
            if IN_COLAB
            else "Copy .env.example to .env at the repository root and fill "
            "them in, then re-run this cell."
        )
        raise RuntimeError(
            "Missing required key(s): " + ", ".join(missing) + ". " + fix
        )
    print("\nAll required keys are present.")


report_keys(REQUIRED_KEYS, OPTIONAL_KEYS)

### How a key reaches a model

`load_dotenv()` put the values into `os.environ`. From there:

- **Direct**: `get_llm_from_name("gemini-3.0-flash")` resolves the alias through
  `LLM_REGISTRY` into a LiteLLM model string, and LiteLLM reads the provider's
  standard variable at call time.
- **OpenRouter**: we build the same `SystemPrefixedLM` the CLI uses, passing
  `api_base` and the OpenRouter key explicitly. `SystemPrefixedLM` is what gives
  us the system-prompt injection and per-call cost tracking, so the OpenRouter
  path is not a downgrade.

The helper below is the only place that knows the difference.

In [ ]:
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.llms import SystemPrefixedLM


def build_lm(role: str, system_prompt: str = "", **model_kwargs):
    """Return a cost-tracking LM for a pipeline role, honouring USE_OPENROUTER.

    Args:
        role: one of "material", "synthesis", "judge", "linker".
        system_prompt: injected at the start of every call.
        **model_kwargs: temperature, max_tokens, ... passed to the LM.
    """
    if USE_OPENROUTER:
        return SystemPrefixedLM(
            system_prompt,
            f"openrouter/{OPENROUTER_MODELS[role]}",
            api_base=OPENROUTER_API_BASE,
            api_key=os.environ["OPENROUTER_API_KEY"],
            **model_kwargs,
        )
    return get_llm_from_name(
        DIRECT_MODELS[role],
        model_kwargs=model_kwargs,
        system_prompt=system_prompt,
    )


def vlm_model_name() -> str:
    """Model string for ClaudeLinePlotDataExtractor, honouring USE_OPENROUTER.

    ClaudeAPIClient builds `anthropic.Anthropic(base_url=...)` without passing
    an explicit key, so the SDK reads ANTHROPIC_API_KEY from the environment.
    On the OpenRouter path we point that variable at the OpenRouter key for
    this session only - nothing is written back to .env.
    """
    if USE_OPENROUTER:
        os.environ["ANTHROPIC_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
        return f"openrouter/{OPENROUTER_MODELS['vlm']}"
    return DIRECT_MODELS["vlm"]


print(
    "material  ->",
    (OPENROUTER_MODELS if USE_OPENROUTER else DIRECT_MODELS)["material"],
)
print(
    "synthesis ->",
    (OPENROUTER_MODELS if USE_OPENROUTER else DIRECT_MODELS)["synthesis"],
)
print("vlm       ->", vlm_model_name())

### Paths and the result cache

Two small helpers used throughout:

- `repo_root()` finds the repository from wherever the notebook is running, so
  nothing depends on the working directory.
- `cached()` runs a function once, writes its JSON result under
  `data/tutorials/<paper_id>/cache/`, and reuses it on every later run. This is
  what makes re-running free.

In [ ]:
import json
from collections.abc import Callable


WORK_DIR = REPO_ROOT / "data" / "tutorials" / PAPER_ID  # data/ is git-ignored
CACHE_DIR = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "results"
PDF_PATH = WORK_DIR / f"{PAPER_ID}.pdf"

for directory in (WORK_DIR, CACHE_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def cached(name: str, compute: Callable):
    """Compute a JSON-serialisable value once, then reuse it on later runs.

    Set FORCE_REFRESH = True in the configuration cell to recompute everything,
    or delete a single file under CACHE_DIR to redo just that step.
    """
    path = CACHE_DIR / f"{name}.json"
    if path.exists() and not FORCE_REFRESH:
        print(f"[cache] reusing {path.name} (delete it to recompute)")
        return json.loads(path.read_text())
    value = compute()
    path.write_text(json.dumps(value, indent=2, default=str))
    print(f"[cache] wrote {path.name}")
    return value


print(f"repo root : {REPO_ROOT}")
print(f"work dir  : {WORK_DIR}")

## Step 1 — Get the paper

The PDF is **not** committed to this repository: it is CC BY-NC-ND, and `data/`
is git-ignored anyway. The cell below downloads it from arXiv into
`data/tutorials/2404.08872/`.

If your network blocks the download, fetch
[arxiv.org/pdf/2404.08872v1](https://arxiv.org/pdf/2404.08872v1) in a browser and
save it to the path the cell prints.

In [ ]:
import urllib.request

if PDF_PATH.exists():
    print(f"Already downloaded: {PDF_PATH} ({PDF_PATH.stat().st_size:,} bytes)")
else:
    print(f"Downloading {PAPER_PDF_URL} ...")
    request = urllib.request.Request(
        PAPER_PDF_URL, headers={"User-Agent": "lemat-synth-tutorial"}
    )
    with urllib.request.urlopen(request) as response:
        PDF_PATH.write_bytes(response.read())
    print(f"Saved {PDF_PATH} ({PDF_PATH.stat().st_size:,} bytes)")

if not PDF_PATH.exists() or PDF_PATH.stat().st_size < 100_000:
    raise RuntimeError(
        f"Download looks wrong. Fetch {PAPER_PDF_URL} manually and save it to "
        f"{PDF_PATH}"
    )

## Step 2 — PDF → markdown

Two extractors implement the same `PdfExtractorInterface` and return markdown
with figures embedded as base64 data URIs:

| Extractor | Cost | Notes |
|-----------|------|-------|
| `MistralPDFExtractor` | API call | Better OCR on multi-column and scanned PDFs |
| `DoclingPDFExtractor` | Free, local | No API key; slower, and heavier on first run |

Switch with `PDF_EXTRACTOR` in the configuration cell. The result is cached, so
you pay for OCR once.

In [ ]:
def extract_pdf_text() -> dict:
    """Run the configured PDF extractor over the paper. Cached."""
    if PDF_EXTRACTOR == "mistral":
        from llm_synthesis.transformers.pdf_extraction import (
            MistralPDFExtractor,
        )

        extractor = MistralPDFExtractor(structured=False)
    else:
        from llm_synthesis.transformers.pdf_extraction import (
            DoclingPDFExtractor,
        )

        extractor = DoclingPDFExtractor(pipeline="standard", format="markdown")

    print(f"Extracting text with {type(extractor).__name__} ...")
    return {"markdown": extractor.forward(PDF_PATH.read_bytes())}


paper_markdown = cached("01_pdf_markdown", extract_pdf_text)["markdown"]

print(f"\n{len(paper_markdown):,} characters of markdown")

In [ ]:
# The markdown keeps figures inline as base64 data URIs - that is what the
# figure extractor later reads. Skip over them for a readable preview.
import re

preview = re.sub(r"!\[[^\]]*\]\(data:image[^)]*\)", "[FIGURE]", paper_markdown)
print(preview[:1500])
print("...")

## Step 3 — Clean the text, build a `Paper`

`clean_text()` strips the embedded figures and the reference list. Both are pure
noise for a language model looking for a synthesis procedure, and the base64
blobs alone would blow the context window.

Note that we keep `paper_markdown` around unchanged — the figure extractor in
Step 6 needs those very images.

In [ ]:
from llm_synthesis.models.paper import Paper
from llm_synthesis.utils import clean_text

clean_paper_text = clean_text(paper_markdown)

paper = Paper(
    id=PAPER_ID,
    name=PAPER_ID,
    publication_text=clean_paper_text,
    si_text="",  # this paper has no supporting information
)

print(f"raw markdown : {len(paper_markdown):,} chars")
print(f"cleaned text : {len(clean_paper_text):,} chars")
print(f"removed      : {len(paper_markdown) - len(clean_paper_text):,} chars")

## Step 4 — Which materials were synthesised?

The material extractor is a `DspyTextExtractor` driven by a signature you write:
instructions, an input description and an output description. Wording matters —
the instruction below insists on listing *variants* separately, because the
common failure is to collapse `1%Ru/CaO`, `3%Ru/CaO` and `5%Ru/CaO` into one
generic `Ru/CaO`.

For this paper we expect exactly two: **MoS₂** and the **MoS₂–rGO** composite.

In [ ]:
from llm_synthesis.transformers.material_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)

material_signature = make_dspy_text_extractor_signature(
    signature_name="TextToMaterials",
    instructions=(
        "Extract ALL distinct material compositions that were synthesized in "
        "this paper. If the paper studies multiple variants of a material "
        "(different loadings, dopant concentrations, or composite ratios), "
        "list EACH variant separately - do NOT merge them into one generic "
        "name. Include only materials actually synthesized, not ones merely "
        "cited or used as a purchased reference."
    ),
    input_description="The publication text to extract synthesized materials from.",
    output_name="materials",
    output_description=(
        "All distinct synthesized materials as a comma-separated list of "
        "chemical formulas or composite names, e.g. 'MoS2, MoS2-rGO'."
    ),
)

# max_tokens matters here: the extractor writes a reasoning trace before the
# list, and a truncated response yields a half-written final material.
material_extractor = DspyTextExtractor(
    signature=material_signature,
    lm=build_lm("material", temperature=0.0, max_tokens=16000),
)


def extract_materials() -> dict:
    """One LLM call: the comma-separated material list. Cached."""
    raw = material_extractor.forward(input=clean_paper_text)
    materials = [
        name.strip()
        for name in raw.replace("\n", ",").split(",")
        if name.strip()
    ]
    return {"raw": raw, "materials": materials}


materials = cached("02_materials", extract_materials)["materials"]

print(f"\n{len(materials)} material(s) found:")
for index, name in enumerate(materials, 1):
    print(f"  {index}. {name}")

## Step 5 — Extract a structured recipe per material

`DspySynthesisExtractor` writes into `GeneralSynthesisOntology`: precursors with
amounts, ordered steps with conditions, equipment. Two details worth noticing:

- The **system prompt repeats the allowed enum values** for `synthesis_method`
  and `target_compound_type`. The schema enforces them, but a model that has not
  been told the list invents values and the record then fails validation.
- The **judge runs immediately after each extraction**, scoring seven dimensions
  1–5 against the source text. Extraction and evaluation are separate concerns
  in this codebase (`transformers/` produces, `metrics/` evaluates) and this is
  where they meet.

In [ ]:
from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)

SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning',
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion',
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth',
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering',
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly',
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation',
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing',
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'metals & alloys', 'ceramics & glasses', 'polymers & soft matter', 'composites',
'semiconductors & electronic', 'nanomaterials', 'two-dimensional materials',
'framework & porous materials', 'biomaterials & biological', 'liquid materials',
'hybrid & organic-inorganic', 'functional materials & catalysts', 'energy & sustainability',
'smart & responsive materials', 'emerging & quantum materials', 'other'

If the exact method is not in the list, use the closest match or 'other'."""

synthesis_extractor = DspySynthesisExtractor(
    signature=make_dspy_synthesis_extractor_signature(
        instructions=(
            "Extract the complete structured synthesis procedure for the "
            "specified material. Include every step, condition (temperature, "
            "time, atmosphere), piece of equipment and precursor. Preserve all "
            "quantitative details."
        )
    ),
    lm=build_lm(
        "synthesis",
        system_prompt=SYNTHESIS_SYSTEM_PROMPT,
        temperature=0.0,
        max_tokens=32000,
    ),
)

judge = DspyGeneralSynthesisJudge(
    signature=make_general_synthesis_judge_signature(),
    lm=build_lm("judge", temperature=0.1, max_tokens=4096),
)

print("[OK] synthesis extractor and judge ready")

In [ ]:
def extract_syntheses() -> list:
    """One extraction + one judge call per material. Cached as plain dicts."""
    records = []
    for index, material in enumerate(materials, 1):
        print(f"\n[{index}/{len(materials)}] {material}")
        synthesis = synthesis_extractor.forward(
            input=(clean_paper_text, material)
        )
        print(
            f"    method={synthesis.synthesis_method} "
            f"steps={len(synthesis.steps)} "
            f"precursors={len(synthesis.starting_materials)}"
        )
        try:
            evaluation = judge.forward(
                (clean_paper_text, json.dumps(synthesis.model_dump()), material)
            )
            print(f"    judge overall: {evaluation.scores.overall_score}/5.0")
            evaluation_dump = evaluation.model_dump()
        except Exception as exc:  # a failed judge must not lose the extraction
            print(f"    [WARN] judge failed: {exc}")
            evaluation_dump = None
        records.append(
            {
                "material": material,
                "synthesis": synthesis.model_dump(),
                "evaluation": evaluation_dump,
            }
        )
    return records


synthesis_records = cached("03_syntheses", extract_syntheses)
print(f"\n[OK] {len(synthesis_records)} recipe(s)")

In [ ]:
# Cached dicts become objects again through the same schema the extractor used.
from llm_synthesis.models.ontologies import GeneralSynthesisOntology

recipes = {
    record["material"]: GeneralSynthesisOntology.model_validate(
        record["synthesis"]
    )
    for record in synthesis_records
}

for material, recipe in recipes.items():
    print("=" * 64)
    print(
        f"{material}  ({recipe.synthesis_method}, {recipe.target_compound_type})"
    )
    print("=" * 64)
    print("  Precursors:")
    for precursor in recipe.starting_materials:
        amount = (
            f"{precursor.amount} {precursor.unit or ''}".strip()
            if precursor.amount is not None
            else "amount not stated"
        )
        print(f"    - {precursor.name}: {amount}")
    print("  Steps:")
    for step in recipe.steps:
        conditions = []
        if step.conditions:
            c = step.conditions
            if c.temperature is not None:
                conditions.append(
                    f"{c.temperature} {c.temp_unit or ''}".strip()
                )
            if c.duration is not None:
                conditions.append(f"{c.duration} {c.time_unit or ''}".strip())
            if c.atmosphere:
                conditions.append(c.atmosphere)
        suffix = f"  [{', '.join(conditions)}]" if conditions else ""
        print(f"    {step.step_number}. {step.action}{suffix}")
    print(
        f"  Equipment: {', '.join(e.name for e in recipe.equipment) or 'none recorded'}"
    )
    print()

## Step 5b — Check against the human ground truth

This paper is in the repository's `annotations/` corpus, so an expert has
already written down the correct recipe by hand. Comparing costs nothing and is
the most direct answer to "is any of this right?".

Expect small, legitimate differences: the human wrote `MOS2`, the model will
write `MoS2`; step counts differ when a model splits "wash and dry" into two
actions. What you are looking for is agreement on **method**, **precursors** and
the **numbers** — 200 °C for 22 h, dried at 80 °C for 8 h.

In [ ]:
import re

GROUND_TRUTH_PATH = REPO_ROOT / "annotations" / PAPER_ID / "result_human.json"


def normalise(name: str) -> str:
    """Lowercase and drop punctuation so 'MOS2-rGO' == 'MoS2-rGO'."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


if GROUND_TRUTH_PATH.exists():
    ground_truth = json.loads(GROUND_TRUTH_PATH.read_text())
    human_by_name = {
        normalise(entry["material_name"]): entry["human_recipe"]
        for entry in ground_truth["materials"]
    }

    print(f"human annotation: {GROUND_TRUTH_PATH.relative_to(REPO_ROOT)}")
    print(
        f"expert found {len(human_by_name)} material(s), we found {len(recipes)}\n"
    )

    for material, recipe in recipes.items():
        human = human_by_name.get(normalise(material))
        print(f"--- {material} ---")
        if human is None:
            print("    no human record under this name (naming mismatch?)")
            continue
        print(
            f"    method     ours={recipe.synthesis_method!r}  human={human.get('synthesis_method')!r}"
        )
        print(
            f"    steps      ours={len(recipe.steps)}  human={len(human.get('steps') or [])}"
        )
        ours = {normalise(p.name) for p in recipe.starting_materials}
        theirs = {
            normalise(p["name"])
            for p in (human.get("starting_materials") or [])
            if p.get("name")
        }
        print(
            f"    precursors both={len(ours & theirs)}  only-ours={len(ours - theirs)}  only-human={len(theirs - ours)}"
        )
        temperatures = sorted(
            {
                s.conditions.temperature
                for s in recipe.steps
                if s.conditions and s.conditions.temperature is not None
            }
        )
        human_temperatures = sorted(
            {
                (s.get("conditions") or {}).get("temperature")
                for s in (human.get("steps") or [])
                if (s.get("conditions") or {}).get("temperature") is not None
            }
        )
        print(f"    temps      ours={temperatures}  human={human_temperatures}")
else:
    print(f"No ground truth at {GROUND_TRUTH_PATH} - skipping comparison.")

> Tutorial 5 turns this one-paper eyeball check into a measurement across the
> whole 36-paper corpus, including how far the LLM judges drift from human
> scores.

## Step 6 — Find and segment the figures

Everything from here on is the performance half. It reads the *raw* markdown,
because that is where the base64 images still live.

Two segmentation backends: **Florence-2** with a LoRA adapter (the library
default: one model, binary quantitative/qualitative classification) and
**Grounding DINO + ResNet-152** (two models, 28 figure classes). Both download
weights on first use.

In [ ]:
if SKIP_PERFORMANCE:
    print("[skip] SKIP_PERFORMANCE = True")
    figures = []
else:
    from llm_synthesis.transformers.figure_extraction import (
        FigureExtractorMarkdown,
    )

    figure_extractor = FigureExtractorMarkdown(
        segmenter="florence",  # or "dino" for 28 granular figure classes
        florence_repo_id="amayuelas/plot-visualization-florence-2-lora-32",
    )
    figures = figure_extractor.forward(paper_markdown)

    print(f"{len(figures)} subfigure(s) segmented")
    for index, figure in enumerate(figures, 1):
        print(
            f"  {index}. {figure.figure_reference or 'unreferenced'}: "
            f"{figure.figure_class} (quantitative={figure.quantitative})"
        )

## Step 7 — Read the data points off the plots

`ClaudeLinePlotDataExtractor` sends each figure to a vision model together with
the surrounding paper text, and asks for the series and their coordinates. The
context matters: chemical formulas are routinely misread from low-resolution
images, and the caption is more reliable than the pixels.

Every figure is sent — the VLM itself decides which ones carry extractable data
and returns nothing for the rest. This is the most expensive step, one call per
figure, so it is cached like everything else.

In [ ]:
from llm_synthesis.models.plot import ExtractedLinePlotData


def extract_plot_data() -> list:
    """One VLM call per figure. Cached. Returns [{figure_index, plot}, ...]."""
    if SKIP_PERFORMANCE or not figures:
        return []

    from llm_synthesis.models.figure import FigureInfoWithPaper
    from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (  # noqa: E501
        ClaudeLinePlotDataExtractor,
    )
    from llm_synthesis.utils.figure_utils import clean_text_from_images

    plot_extractor = ClaudeLinePlotDataExtractor(model_name=vlm_model_name())
    paper_text_for_vlm = clean_text_from_images(paper_markdown)

    records = []
    for index, figure in enumerate(figures):
        label = figure.figure_reference or f"figure {index}"
        print(f"  [{index + 1}/{len(figures)}] {label} ...", end=" ")
        figure_with_paper = FigureInfoWithPaper(
            base64_data=figure.base64_data,
            alt_text=figure.alt_text,
            position=figure.position,
            context_before=figure.context_before,
            context_after=figure.context_after,
            figure_reference=figure.figure_reference,
            figure_class=figure.figure_class,
            quantitative=figure.quantitative,
            paper_text=paper_text_for_vlm,
            si_text="",
        )
        try:
            plot = plot_extractor.forward(figure_with_paper)
        except Exception as exc:
            print(f"failed: {exc}")
            continue
        if plot and plot.name_to_coordinates:
            records.append({"figure_index": index, "plot": plot.model_dump()})
            print(f"{len(plot.name_to_coordinates)} series")
        else:
            print("no extractable data")

    print(f"\ncost so far: ${plot_extractor.get_cost():.4f}")
    return records


plot_records = cached("04_plot_data", extract_plot_data)

plots = [ExtractedLinePlotData.model_validate(r["plot"]) for r in plot_records]
plot_figure_indices = [r["figure_index"] for r in plot_records]
print(f"\n{len(plots)} plot(s) with data")

In [ ]:
for position, plot in enumerate(plots):
    print(f"Plot {position}: {plot.title or 'untitled'}")
    print(f"  x: {plot.x_axis_label} [{plot.x_axis_unit}]")
    print(f"  y: {plot.y_left_axis_label} [{plot.y_left_axis_unit}]")
    for series_name, coordinates in plot.name_to_coordinates.items():
        print(f"    - {series_name}: {len(coordinates)} points")

## Step 8 — Keep only the plots that mean something

A paper's figures include XRD patterns, FTIR spectra, SEM images and the
performance curves you actually want. `PlotFilterConfig` decides which is which
from the axis labels and units, and it is the single knob you turn when moving
to a new domain:

| Preset | Keeps |
|--------|-------|
| `for_catalysis()` (default) | conversion / yield / selectivity vs temperature |
| `for_electrochemistry()` | current / capacitance vs potential |
| `for_superconductivity()` | resistance / resistivity vs temperature, with a veto list for derivatives and difference plots |
| `no_filter()` | everything |

Our paper reports **LSV curves** — current density against potential — so
`for_electrochemistry()` is the right preset. The constructor takes plain
keyword lists if none of the presets fit.

In [ ]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import (
    PlotFilter,
)

filter_config = PlotFilterConfig.for_electrochemistry()

# A custom domain looks like this:
# filter_config = PlotFilterConfig(
#     x_axis_labels=["potential", "voltage"],
#     x_axis_units=["v", "mv"],
#     y_axis_keywords=["current density", "overpotential"],
#     y_axis_units=["ma/cm2", "ma cm-2"],
# )

plot_filter = PlotFilter(filter_config)

print(f"x-axis labels : {filter_config.x_axis_labels}")
print(f"x-axis units  : {filter_config.x_axis_units}")
print(f"y-axis keywords: {filter_config.y_axis_keywords}")
print(f"y-axis units  : {filter_config.y_axis_units}\n")

if plots:
    relevant_plots, skip_counts = plot_filter.filter_plots(
        plots, log_skipped=False
    )
    print(f"{len(relevant_plots)} of {len(plots)} plot(s) kept")
    print(
        f"  skipped, x-axis not relevant: {skip_counts.get('not_relevant_x', 0)}"
    )
    print(
        f"  skipped, y-axis not relevant: {skip_counts.get('not_relevant_y', 0)}"
    )
    print(f"  skipped, no series:           {skip_counts.get('no_series', 0)}")
    for index, plot in relevant_plots:
        print(
            f"    plot {index}: {plot.title or 'untitled'} "
            f"({len(plot.name_to_coordinates)} series)"
        )
else:
    relevant_plots = []
    print("no plots to filter")

### Look at what the VLM read

Worth doing at least once: the digitised series next to the figure it came from.
A vision model reading a plot is the least reliable step in this pipeline, and
this is the cheapest way to calibrate how much you trust it.

In [ ]:
import base64
import io

import matplotlib.pyplot as plt
from PIL import Image

if relevant_plots:
    plot_position, plot = relevant_plots[0]
    figure = figures[plot_figure_indices[plot_position]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    image = Image.open(io.BytesIO(base64.b64decode(figure.base64_data)))
    axes[0].imshow(image)
    axes[0].axis("off")
    axes[0].set_title(f"original: {figure.figure_reference or 'figure'}")

    for series_name, coordinates in plot.name_to_coordinates.items():
        if coordinates:
            axes[1].plot(
                [point[0] for point in coordinates],
                [point[1] for point in coordinates],
                "o-",
                markersize=3,
                label=series_name,
            )
    axes[1].set_xlabel(f"{plot.x_axis_label or 'x'} [{plot.x_axis_unit or ''}]")
    axes[1].set_ylabel(
        f"{plot.y_left_axis_label or 'y'} [{plot.y_left_axis_unit or ''}]"
    )
    axes[1].set_title("digitised by the VLM")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("no relevant plots to display")

## Step 9 — Link each curve to a material

The last gap: a plot series is called `MoS2-rGO` or just `(b)`, and we need to
know which of *our* extracted materials it belongs to. `SeriesMaterialLinker`
asks an LLM to match series names against the material list, given the figure
caption as context.

The prompt is deliberately conservative — baselines, reference electrodes and
equilibrium lines must stay unmatched, and an empty result is a valid answer.
A wrong link is worse than a missing one, because nothing downstream can detect
it.

In [ ]:
import time

from llm_synthesis.models.performance import PlotMaterialMapping
from llm_synthesis.transformers.performance_linking.base import LinkingInput
from llm_synthesis.transformers.performance_linking.series_material_linker import (
    SeriesMaterialLinker,
)


def link_series_to_materials() -> list:
    """One LLM call per relevant plot. Cached."""
    if SKIP_PERFORMANCE or not relevant_plots:
        return []

    linker = SeriesMaterialLinker(
        lm=build_lm("linker", temperature=0.0, max_tokens=8000)
    )

    records = []
    for plot_position, plot in relevant_plots:
        figure = figures[plot_figure_indices[plot_position]]
        series_names = list(plot.name_to_coordinates.keys())
        print(f"plot {plot_position}: {series_names}")

        mappings = linker.forward(
            LinkingInput(
                materials=materials,
                series_names=series_names,
                context=f"{figure.context_before} {figure.context_after}",
                plot_metadata={
                    "title": plot.title,
                    "x_axis_label": plot.x_axis_label,
                    "x_axis_unit": plot.x_axis_unit,
                    "y_left_axis_label": plot.y_left_axis_label,
                    "y_left_axis_unit": plot.y_left_axis_unit,
                },
            )
        )
        matched = {mapping.series_name for mapping in mappings}
        for mapping in mappings:
            print(
                f"    '{mapping.series_name}' -> '{mapping.material_name}' "
                f"({mapping.confidence})"
            )
        unmatched = [name for name in series_names if name not in matched]
        if unmatched:
            print(f"    unmatched: {unmatched}")

        records.append(
            PlotMaterialMapping(
                plot_index=plot_position,
                figure_reference=figure.figure_reference,
                mappings=mappings,
                unmatched_series=unmatched,
            ).model_dump()
        )

        time.sleep(1)  # avoid rate-limiting on OpenRouter
    return records


mapping_records = cached("05_linking", link_series_to_materials)
plot_mappings = [PlotMaterialMapping.model_validate(r) for r in mapping_records]
print(f"\n{len(plot_mappings)} plot(s) linked")

## Step 10 — Aggregate, then judge the linking

`aggregate_all_materials_performance` inverts the mapping: instead of "this plot
has these series", you get "this material has these performance curves". That
per-material view is what gets written to disk.

The linking judge then scores the result on four criteria and checks nine
specific failure modes — name mismatches, precursor mistaken for product,
dual-axis confusion, false positives and negatives.

In [ ]:
from llm_synthesis.utils.performance_utils import (
    aggregate_all_materials_performance,
)

if plot_mappings and plots:
    performance_data = aggregate_all_materials_performance(
        materials, plot_mappings, plots
    )
else:
    performance_data = {}

for material in materials:
    entry = performance_data.get(material)
    if entry is None:
        print(f"{material}: no performance data linked")
        continue
    print(f"{material}: {len(entry.plot_data)} curve(s)")
    for curve in entry.plot_data:
        print(
            f"    '{curve.series_name}' - {curve.y_axis_label} "
            f"[{curve.y_axis_unit}], {len(curve.coordinates)} points, "
            f"confidence {curve.confidence}"
        )

In [ ]:
from llm_synthesis.metrics.judge.linking_judge import (
    DspyLinkingJudge,
    make_linking_judge_signature,
)


def judge_linking() -> dict | None:
    """One LLM call scoring the whole linking result. Cached."""
    if not (plot_mappings and performance_data):
        return None

    linking_judge = DspyLinkingJudge(
        signature=make_linking_judge_signature(),
        lm=build_lm("judge", temperature=0.1, max_tokens=4096),
    )
    evaluation = linking_judge.forward(
        (
            clean_paper_text,
            json.dumps(synthesis_records, default=str),
            json.dumps([plot.model_dump() for plot in plots], default=str),
            json.dumps(
                {
                    "mappings": [m.model_dump() for m in plot_mappings],
                    "performance_per_material": {
                        name: entry.model_dump()
                        for name, entry in performance_data.items()
                    },
                },
                default=str,
            ),
        )
    )
    return evaluation.model_dump()


linking_evaluation = cached("06_linking_judge", judge_linking)

if linking_evaluation:
    scores = linking_evaluation["scores"]
    print("Linking evaluation")
    for key, value in scores.items():
        if key.endswith("_score"):
            print(
                f"  {key.replace('_score', '').replace('_', ' '):<30} {value}/5.0"
            )
    active = [
        flag
        for flag, value in linking_evaluation["failure_flags"].items()
        if value
    ]
    print(f"  failure flags: {active or 'none'}")
else:
    print("nothing to judge (no linked performance data)")

## Step 11 — Save, in the layout the CLI produces

One JSON per material, plus the plot mappings and a summary. This is exactly
what `lemat-synth extract … with_performance=true` writes, so anything that
consumes CLI output consumes this too.

In [ ]:
from llm_synthesis.utils.performance_utils import sanitize_filename

paper_output_dir = OUTPUT_DIR / PAPER_ID
paper_output_dir.mkdir(parents=True, exist_ok=True)

for record in synthesis_records:
    material = record["material"]
    entry = performance_data.get(material)
    payload = {
        "material": material,
        "synthesis": record["synthesis"],
        "evaluation": record["evaluation"],
        "performance": entry.model_dump() if entry else None,
    }
    path = paper_output_dir / f"{sanitize_filename(material)}.json"
    path.write_text(json.dumps(payload, indent=2, default=str))

(paper_output_dir / "performance_mappings.json").write_text(
    json.dumps([m.model_dump() for m in plot_mappings], indent=2, default=str)
)

summary = {
    "paper_id": PAPER_ID,
    "paper_url": PAPER_PDF_URL,
    "total_materials": len(materials),
    "materials_list": materials,
    "materials_with_performance_list": [
        m for m in materials if m in performance_data
    ],
    "materials_without_performance_list": [
        m for m in materials if m not in performance_data
    ],
    "total_plots_extracted": len(plots),
    "plots_linked": len(plot_mappings),
    "linking_evaluation": linking_evaluation,
}
(paper_output_dir / "linking_summary.json").write_text(
    json.dumps(summary, indent=2, default=str)
)

print(f"written to {paper_output_dir}")
for path in sorted(paper_output_dir.iterdir()):
    print(f"  {path.name:<40} {path.stat().st_size:>8,} bytes")

## What's next

- **[Tutorial 5 — Evaluating extraction quality](05_evaluating_extraction_quality.ipynb)**:
  the ground-truth check from Step 5b, done properly across 36 papers.
- **[Tutorial 6 — Customising the ontology](06_customizing_the_ontology.ipynb)**:
  when the schema does not capture what your domain cares about.
- **[Tutorial 3 — Batch extraction with the CLI](03_batch_extraction_with_the_cli.ipynb)**:
  everything above as one command over a folder of papers, with the same output
  layout.

### Running this on your own paper

Three things to change:

1. `PAPER_ID` and `PAPER_PDF_URL` — or drop a PDF into
   `data/tutorials/<your_id>/<your_id>.pdf` yourself and skip Step 1.
2. The `PlotFilterConfig` preset in Step 8, to match the axes your field plots.
3. The `"material"` model, if your paper has many closely related compositions —
   `gemini-3.0-pro` is meaningfully better at that one job.

Delete `data/tutorials/<paper_id>/cache/` to start a clean run, or set
`FORCE_REFRESH = True` to recompute everything in place.
